# 🚀 Setup del entorno — GitHub Codespaces (ML/DL + Data Science)

Este notebook configura el entorno **compartido** del proyecto para que todo el equipo trabaje sobre la misma base. Genera los archivos de configuración del **devcontainer** que GitHub Codespaces usa para construir el entorno.

## ¿Por qué este enfoque?

En un Codespace el propio entorno **ya es** un contenedor. No tiene sentido correr `docker compose` dentro — sería un contenedor dentro de otro contenedor (y se llena el disco, como ya vimos). En vez de eso, le decimos a GitHub *cómo* construir el contenedor del Codespace mediante `.devcontainer/devcontainer.json`.

**Ventaja para el equipo**: cualquier persona que abra el Codespace del repo obtiene exactamente el mismo entorno, con las mismas librerías, mismas extensiones de VS Code, misma versión de Python. Cero "a mí me funciona".

## Stack

- **Python 3.13** (imagen oficial devcontainer de Microsoft)
- **Data Science**: NumPy, Pandas, SciPy, Matplotlib, Seaborn, Plotly
- **ML clásico**: scikit-learn, XGBoost, LightGBM, statsmodels
- **Deep Learning**: PyTorch (CPU only — desde el índice oficial de PyTorch para no traer ~3 GB de CUDA)
- **Jupyter** integrado en VS Code (no hace falta levantar servidor)

## Estructura que vamos a crear

```
proyecto/
├── .devcontainer/
│   └── devcontainer.json   ← config del Codespace
├── 00_setup.ipynb          ← este notebook
├── requirements.txt
├── .gitignore
├── notebooks/
├── data/
└── src/
```

---
## 1. Estructura de carpetas

In [ ]:
import os

for carpeta in [".devcontainer", "notebooks", "data", "src"]:
    os.makedirs(carpeta, exist_ok=True)

# .gitkeep para que git tracking las carpetas vacías de trabajo
for carpeta in ["notebooks", "src"]:
    open(os.path.join(carpeta, ".gitkeep"), "a").close()

print("✅ Estructura creada")
!ls -la

---
## 2. `.devcontainer/devcontainer.json`

Este es el corazón de la configuración. Define:

- **Imagen base**: `mcr.microsoft.com/devcontainers/python:1-3.13-bookworm` (Python 3.13 + utilidades ya instaladas).
- **Extensiones de VS Code** que se instalan automáticamente para todos.
- **`postCreateCommand`**: lo que se ejecuta una vez al crear el Codespace — instala las dependencias del proyecto.

Nota importante: instalamos **`torch` desde el índice CPU de PyTorch** (`download.pytorch.org/whl/cpu`) para evitar que se traiga las librerías de CUDA (~3 GB que no usamos en CPU).

In [ ]:
%%writefile .devcontainer/devcontainer.json
{
  "name": "Proyecto ML/DS",
  "image": "mcr.microsoft.com/devcontainers/python:1-3.13-bookworm",

  "features": {
    "ghcr.io/devcontainers/features/git:1": {}
  },

  "customizations": {
    "vscode": {
      "extensions": [
        "ms-python.python",
        "ms-python.vscode-pylance",
        "ms-toolsai.jupyter",
        "ms-toolsai.jupyter-keymap",
        "ms-toolsai.jupyter-renderers",
        "ms-toolsai.vscode-jupyter-cell-tags",
        "ms-toolsai.vscode-jupyter-slideshow",
        "charliermarsh.ruff"
      ],
      "settings": {
        "python.defaultInterpreterPath": "/usr/local/bin/python",
        "jupyter.notebookFileRoot": "${workspaceFolder}",
        "[python]": {
          "editor.defaultFormatter": "charliermarsh.ruff",
          "editor.formatOnSave": true
        }
      }
    }
  },

  "postCreateCommand": "pip install --upgrade pip && pip install --index-url https://download.pytorch.org/whl/cpu torch~=2.5 && pip install -r requirements.txt",

  "remoteUser": "vscode"
}


---
## 3. `requirements.txt`

Sin `torch` aquí (se instala en el `postCreateCommand` desde el índice CPU). Dejé TensorFlow comentado: si alguien del grupo lo va a usar, lo descomentan.

In [ ]:
%%writefile requirements.txt
# === Núcleo numérico ===
numpy~=2.1
pandas~=2.2
scipy~=1.14

# === Visualización ===
matplotlib~=3.9
seaborn~=0.13
plotly~=5.24

# === ML clásico ===
scikit-learn~=1.5
xgboost~=2.1
lightgbm~=4.5
statsmodels~=0.14

# === Deep Learning (CPU) ===
# torch se instala desde el índice CPU vía postCreateCommand del devcontainer
# Descomenta la siguiente línea si el equipo decide usar TensorFlow también
# tensorflow-cpu~=2.18

# === Jupyter (kernel; la UI viene en VS Code vía extensiones) ===
ipykernel~=6.29
ipywidgets~=8.1

# === Utilidades ===
tqdm~=4.66
joblib~=1.4
python-dotenv~=1.0


---
## 4. `.gitignore`

Estándar para proyectos Python + datos. Importante: **ignoramos `data/`** para no commitear datasets pesados al repo.

In [ ]:
%%writefile .gitignore
# Datos y modelos
data/*
!data/.gitkeep
models/
*.csv
*.parquet
*.h5
*.pkl
*.pt
*.pth
*.ckpt

# Python
__pycache__/
*.py[cod]
*$py.class
*.egg-info/
.pytest_cache/
.mypy_cache/
.ruff_cache/

# Jupyter
.ipynb_checkpoints/

# Entornos virtuales (por si alguien trabaja en local también)
.venv/
venv/
env/

# Secretos
.env
*.key

# IDE
.vscode/*
!.vscode/settings.json
!.vscode/extensions.json
.idea/

# SO
.DS_Store
Thumbs.db


In [ ]:
# Crear el .gitkeep de data/ para que la carpeta exista en el repo aunque esté vacía
open("data/.gitkeep", "a").close()
print("✅ data/.gitkeep creado")

---
## 5. Verificar lo generado

In [ ]:
!ls -la && echo '---' && ls -la .devcontainer/

---
## 6. 🔄 Flujo para el equipo

### Quien hace el setup inicial (una sola vez)

1. Ejecuta este notebook (genera los archivos).
2. Hace commit y push de **todo** lo generado:
   ```bash
   git add .devcontainer/ requirements.txt .gitignore notebooks/ src/ data/.gitkeep 00_setup.ipynb
   git commit -m "feat: configuración inicial del Codespace (devcontainer + dependencias)"
   git push
   ```
3. **Reconstruir el Codespace actual** para que tome la nueva config:
   - `Ctrl+Shift+P` (o `Cmd+Shift+P`) → escribe `Codespaces: Rebuild Container` → enter
   - Tarda unos 3–5 minutos la primera vez (instala PyTorch y todo lo demás).

### Los demás miembros del equipo

1. Van al repo en GitHub.
2. Click en el botón verde **Code** → pestaña **Codespaces** → **Create codespace on main**.
3. GitHub levanta el Codespace y ejecuta automáticamente el `postCreateCommand` (instala todo).
4. Cuando termine, ya pueden abrir cualquier `.ipynb` en `notebooks/` y ejecutar celdas. VS Code detecta el kernel de Python automáticamente.

### Si alguien agrega una dependencia nueva

1. La añade a `requirements.txt`.
2. Commit + push.
3. Avisa al equipo: cada uno hace **Rebuild Container** en su Codespace para tenerla.

> 💡 Alternativa rápida sin rebuild: `pip install nombre-paquete` en la terminal del Codespace. Pero recuerda añadirla a `requirements.txt` después o se pierde la próxima vez que se reconstruya.

---
## 7. Verificar que todo funciona (después del rebuild)

Después de reconstruir el Codespace, abre un notebook nuevo en `notebooks/` y corre esta celda para confirmar el entorno:

In [1]:
import sys
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import seaborn as sns
import plotly
import torch
import xgboost as xgb
import lightgbm as lgb

print(f"Python      : {sys.version.split()[0]}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"SciPy       : {scipy.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"Matplotlib  : {matplotlib.__version__}")
print(f"Seaborn     : {sns.__version__}")
print(f"Plotly      : {plotly.__version__}")
print(f"XGBoost     : {xgb.__version__}")
print(f"LightGBM    : {lgb.__version__}")
print(f"PyTorch     : {torch.__version__}  (CUDA: {torch.cuda.is_available()}, esperado: False)")

Python      : 3.13.5
NumPy       : 2.4.6
Pandas      : 2.3.3
SciPy       : 1.17.1
scikit-learn: 1.9.0
Matplotlib  : 3.10.9
Seaborn     : 0.13.2
Plotly      : 5.24.1
XGBoost     : 2.1.4
LightGBM    : 4.6.0
PyTorch     : 2.12.0+cpu  (CUDA: False, esperado: False)


---
## 8. Limpieza del intento anterior

Si todavía tienes los archivos del intento con Docker Compose en el repo, ya **no los necesitas**. Bórralos:

```bash
rm -f Dockerfile docker-compose.yml .dockerignore
git add -A
git commit -m "chore: eliminar config Docker Compose (reemplazada por devcontainer)"
git push
```

Y también limpia el espacio que ocupó el build fallido:

```bash
docker system prune -af
```